# Compare DQN Variants (from training_notes.txt)

Notebook này đọc `logs/<ALGO>/training_notes.txt` để tổng hợp:
- **Training Time** (tổng phút, cộng các lần train/gen)
- **Baseline Avg Time** (Random)
- **Final Avg Time** (final model được chọn)
- **Improve (%)** so với baseline: `Improve = (baseline - final) / baseline * 100`
  - Nếu avg time giảm → Improve dương (cải tiến)
  - Nếu avg time tăng → Improve âm (worse)


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
LOG_ROOT = os.path.join(ROOT, 'logs')

# Map tên thuật toán (hiển thị) -> danh sách folder có thể có trong logs/
FOLDER_CANDIDATES = {
    'DQN': ['DQN'],
    'DDQN': ['DDQN'],
    'Dueling': ['Dueling', 'DuelDQN', 'Duel_DQN', 'DuelDQNAgent', 'DuelDQN'],
    'PerDQN': ['PerDQN', 'PER_DQN', 'PERDQN'],
    'MultiStepDQN': ['MultiStepDQN', 'MultiStep_DQN', 'MultiStep'],
    'Rainbow': ['Rainbow'],
}

# Regex parser
RE_BASELINE_1 = re.compile(r'Random\s*\(Baseline\)\s*:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_BASELINE_2 = re.compile(r'Baseline\s*\(Random\)\s*:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_GEN_HEADER = re.compile(r'\bGen\s+(\d+)\s*:', re.IGNORECASE)
RE_FINAL = re.compile(r'Final Model:\s*(.*?)\s*\(from ep=.*?\)\s*\|\s*Avg Time:\s*([0-9]+(?:\.[0-9]+)?)', re.IGNORECASE)
RE_FINAL_GEN = re.compile(r'gen(\d+)', re.IGNORECASE)
RE_TRAIN_MIN = re.compile(r'Training Time:\s*([0-9]+(?:\.[0-9]+)?)\s*minutes', re.IGNORECASE)


def find_algo_dir(algo_display_name: str):
    for folder in FOLDER_CANDIDATES.get(algo_display_name, []):
        p = os.path.join(LOG_ROOT, folder)
        if os.path.isdir(p):
            return p
    return None


def parse_training_notes(text: str):
    # Baseline: lấy match đầu tiên
    baseline = None
    m = RE_BASELINE_1.search(text)
    if m:
        baseline = float(m.group(1))
    else:
        m = RE_BASELINE_2.search(text)
        if m:
            baseline = float(m.group(1))

    # Gen headers (để suy ra số gen nếu file không có final)
    gen_headers = [int(x) for x in RE_GEN_HEADER.findall(text)]

    # Final avg time theo gen (lấy last nếu trùng gen)
    final_by_gen = {}
    for fname, avg_str in RE_FINAL.findall(text):
        gm = RE_FINAL_GEN.search(fname)
        if not gm:
            continue
        gen = int(gm.group(1))
        final_by_gen[gen] = float(avg_str)

    # Training time: trong notes hiện tại không gắn gen trực tiếp,
    # nên giả định thứ tự xuất hiện tương ứng Gen 1..N
    train_mins = [float(x) for x in RE_TRAIN_MIN.findall(text)]
    train_by_gen = {i + 1: v for i, v in enumerate(train_mins)}

    # Max gen
    max_gen = None
    candidates = []
    if final_by_gen:
        candidates.append(max(final_by_gen.keys()))
    if train_by_gen:
        candidates.append(max(train_by_gen.keys()))
    if gen_headers:
        candidates.append(max(gen_headers))

    if candidates:
        max_gen = int(max(candidates))

    return baseline, train_by_gen, final_by_gen, max_gen


rows_summary = []
rows_train_long = []
rows_presence = []

for algo in FOLDER_CANDIDATES.keys():
    algo_dir = find_algo_dir(algo)
    note_path = os.path.join(algo_dir, 'training_notes.txt') if algo_dir else None

    note_found = bool(note_path and os.path.exists(note_path))
    baseline = None
    train_by_gen = {}
    final_by_gen = {}
    max_gen = None

    if note_found:
        with open(note_path, 'r', encoding='utf-8') as f:
            text = f.read()
        baseline, train_by_gen, final_by_gen, max_gen = parse_training_notes(text)

    # Summary: lấy final của max_gen
    final_avg = None
    if max_gen is not None:
        final_avg = final_by_gen.get(max_gen)

    improve_pct = None
    if baseline is not None and final_avg is not None and baseline != 0:
        improve_pct = (baseline - final_avg) / baseline * 100.0

    rows_summary.append({
        'Algorithm': algo,
        'NotesFound': note_found,
        'NotesPath': note_path if note_path else '',
        'MaxGen': max_gen,
        'BaselineAvgTime': baseline,
        'FinalAvgTime': final_avg,
        'ImprovePct': improve_pct,
        'TrainTimeTotalMinutes': float(np.sum(list(train_by_gen.values()))) if train_by_gen else None,
    })

    # Long training time by gen
    for gen, mins in sorted(train_by_gen.items()):
        rows_train_long.append({
            'Algorithm': algo,
            'Gen': int(gen),
            'TrainTimeMinutes': float(mins),
        })

    # Presence by gen (will expand later after we know global max)
    rows_presence.append({
        'Algorithm': algo,
        'MaxGen': max_gen,
    })


df_summary = pd.DataFrame(rows_summary)
df_train_long = pd.DataFrame(rows_train_long)

# Build gen presence wide table
max_gen_global = int(df_summary['MaxGen'].dropna().max()) if df_summary['MaxGen'].notna().any() else 0

presence_rows = []
for _, r in df_summary.iterrows():
    algo = r['Algorithm']
    max_gen = r['MaxGen']

    row = {'Algorithm': algo, 'MaxGen': max_gen}
    for g in range(1, max_gen_global + 1):
        row[f'Gen{g}'] = 'Yes' if (max_gen is not None and g <= int(max_gen)) else '_'
    presence_rows.append(row)

df_presence = pd.DataFrame(presence_rows)

(df_summary, df_presence, df_train_long)

In [ ]:
def fmt_improve(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return 'N/A'
    if x >= 0:
        return f'Improve +{x:.2f}%'
    return f'Worse {x:.2f}%'

print('=== Summary (baseline vs final) ===')
display_summary = df_summary.copy()
display_summary['Improve(%)'] = display_summary['ImprovePct'].apply(fmt_improve)
display_summary = display_summary.drop(columns=['ImprovePct'])
display_summary

print('\n=== Gen presence table ===')
df_presence

In [ ]:
# 1) Improve plot (baseline vs final)
plot_df = df_summary[df_summary['NotesFound']].copy()
plot_df = plot_df.dropna(subset=['ImprovePct'])

if len(plot_df) == 0:
    print('Chưa đủ dữ liệu để vẽ Improve(%): không parse được baseline/final từ training_notes.txt.')
else:
    plot_df = plot_df.sort_values('ImprovePct', ascending=False)

    plt.figure(figsize=(10, 4))
    plt.bar(plot_df['Algorithm'], plot_df['ImprovePct'])
    plt.axhline(0, color='black', linewidth=1)
    plt.ylabel('Improve vs Baseline (%)')
    plt.title('So sánh hiệu năng (Avg Time) so với Baseline')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


# 2) Training time per Gen plot
if df_train_long.empty:
    print('Không có dữ liệu TrainTime theo Gen để vẽ.')
else:
    max_gen = int(df_train_long['Gen'].max())

    for g in range(1, max_gen + 1):
        sub = df_train_long[df_train_long['Gen'] == g].copy()
        if len(sub) == 0:
            continue

        sub = sub.sort_values('TrainTimeMinutes', ascending=False)

        plt.figure(figsize=(10, 4))
        plt.bar(sub['Algorithm'], sub['TrainTimeMinutes'])
        plt.ylabel('Training Time (minutes)')
        plt.title(f'Training time comparison - Gen {g}')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()